# Chapter 11 Data Analysis

**1. Regression**

In [ ]:
!python --version

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats
import pandas 

df = pandas.read_csv('../input/mydata/data.csv')
print(df)
x = df['Time']
y = df['Voltage']
print(x)
print(y)

slope, intercept, r, p, std_err = stats.linregress(x, y)
# https://docs.scipy.org/doc/scipy-1.6.2/reference/generated/scipy.stats.linregress.html

print("slope: ", slope)
print("intercept: ", intercept)
print("std_err: ", std_err)

# y = a*x + b 
def myfunc(x):
  return slope * x + intercept

mymodel = list(map(myfunc, x))

plt.scatter(x, y,label="data")
plt.plot(x, mymodel, "r",label="fitted line")
plt.xlabel("Time")
plt.ylabel("Voltage")
plt.legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from scipy import stats
import pandas 

df = pandas.read_csv('../input/mydata/data.csv')
print(df)
x1 = df['Time'].values.reshape(-1,1)
y1 = df['Voltage'].values.reshape(-1,1)
print(x1)
print(y1)

# https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html
from sklearn.svm import SVR
lr = SVR(kernel = 'linear', C =1000.0)
pr = SVR(kernel = 'poly', C =1000.0, degree = 2)
rr = SVR(kernel = 'rbf', C =1000.0, gamma = 0.85)
lr.fit(x1,y1)
pr.fit(x1,y1)
rr.fit(x1,y1)

plt.figure()
plt.scatter(x1, y1, color='r', label='Data')
plt.plot(x1, lr.predict(x1),label='linear SVR')
plt.plot(x1, pr.predict(x1),label='poly SVR')
plt.plot(x1, rr.predict(x1),label='rbf SVR')
plt.legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd
import numpy as np

#pip install html5lib
#df = pd.read_html('https://en.wikipedia.org/wiki/World_population')
df = pd.read_html('http://www.stats.gov.cn/sj/zxfb/202302/t20230228_1919011.html')
print(f'Total tables: {len(df)}')
print(df[0])
#df = pd.read_html('https://en.wikipedia.org/wiki/World_population', match='Global annual population growth')
#df =df[0]

#x = df['Year'].astype('float')
#y = df['Population'].astype('float')

#plt.scatter(x, y)
#plt.show()


In [ ]:
#pip install requests beautifulsoup4 
from bs4 import BeautifulSoup
import requests
import csv

#url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"
url = "https://en.wikipedia.org/wiki/World_population"
soup = BeautifulSoup(requests.get(url).text, 'html.parser')

tables = soup.find_all('table', class_='sortable')
print(len(tables))

for table in tables:
    ths = table.find_all('th')
    headings = [th.text.strip() for th in ths]
    #print(headings)

table = tables[4]
population = []
for tr in table.find_all('tr'):
    tds = tr.find_all('td')
    #print(tds)        
    if not tds:
        continue
    population.append(tds[0].text.replace('\n', ' ').strip())
print(population)


In [ ]:
# Yahoo Finance seems not working
# pip install requests beautifulsoup4 
import pandas as pd
from bs4 import BeautifulSoup
import requests

def get_stock(t):
    url = f'http://finance.yahoo.com/quote/{t}?p={t}'
    res = requests.get(url)
    soup = (BeautifulSoup(res.content, 'lxml'))
    table = soup.find_all('table')[0]
    labels, data = pd.read_html(str(table))[0].values.T
    return pd.Series(data, labels, name = t)



stock_name = ["AAPL","AMZN","GOOG","TSLA"]

df = pd.concat(map(get_stock, stock_name), axis=1)
print(df)
print(df.T)

In [ ]:
import yfinance as yf

def get_stock_price(symbol):
    stock = yf.Ticker(symbol)
    #history = stock.history(period="1d")
    history = stock.history(start="2017-01-01", end="2023-04-30")
    #return history['Close'].iloc[-1]
    return history['Close']

symbol = "AAPL"  # Replace with the stock symbol you want
price = get_stock_price(symbol)
print(f"The current price of {symbol} is {price}")

In [ ]:
fig, ax = plt.subplots(figsize=(16,9))
ax.plot(price.index, price, label=symbol)

ax.set_xlabel('Date')
ax.set_ylabel('Closing price ($)')
ax.legend()
plt.show()

In [ ]:
# Partial Least Squares (PLS) regression 
# https://nirpyresearch.com
# https://github.com/nevernervous78/nirpyresearch
# https://www.ibm.com/docs/en/spss-statistics/SaaS?topic=features-partial-least-squares-regression

In [ ]:
from sys import stdout
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/nevernervous78/nirpyresearch/master/data/milk-powder.csv')
#data = pd.read_csv('https://raw.githubusercontent.com/nevernervous78/nirpyresearch/master/data/milk.csv')
#data = pd.read_csv('https://raw.githubusercontent.com/nevernervous78/nirpyresearch/master/data/plums.csv')
print(data.head())
print(data.tail())
y = pd.DataFrame.to_numpy(data)[:,1].astype('uint8')
X = pd.DataFrame.to_numpy(data)[:,2:]
# wavelength range in nm
wl = np.linspace(350,2500, num=X.shape[1], endpoint=True)
# wl = np.arange(1100,2300,2) # wavelengths

import matplotlib.pyplot as plt
plt.plot(wl,X.T)
plt.ylabel('Absorption')
plt.title('NIR Spectra')
plt.figure()
plt.plot(y)
plt.title('Milk %')
plt.ylabel('Measured Values')
plt.show()


In [ ]:
# Calculate second derivative
X2 = savgol_filter(X, 17, polyorder = 2,deriv=2) #deriv = 0, 1, 2 (original, first, second derivative)
# Plot second derivative
plt.figure(figsize=(8,4.5))
with plt.style.context(('ggplot')):
  plt.plot(X2.T)
  plt.xlabel('Wavelength (nm)')
  plt.ylabel('D2 Absorbance')
  plt.show()


In [ ]:
def optimise_pls_cv(X, y, n_comp, plot_components=True):
    '''Run PLS including a variable number of components, up to n_comp,
       and calculate MSE '''
    mse = []
    component = np.arange(1, n_comp)
    for i in component:
        pls = PLSRegression(n_components=i)
        # Cross-validation
        y_cv = cross_val_predict(pls, X, y, cv=10)
        mse.append(mean_squared_error(y, y_cv))
        comp = 100*(i+1)/40
        # Trick to update status on the same line
        stdout.write("\r%d%% completed" % comp)
        stdout.flush()
    stdout.write("\n")
    # Calculate and print the position of minimum in MSE
    msemin = np.argmin(mse)
    print("Suggested number of components: ", msemin+1)
    stdout.write("\n")
    if plot_components is True:
        with plt.style.context(('ggplot')):
            plt.plot(component, np.array(mse), '-v', color = 'blue', mfc='blue')
            plt.plot(component[msemin], np.array(mse)[msemin], 'P', ms=10, mfc='red')
            plt.xlabel('Number of PLS components')
            plt.ylabel('MSE')
            plt.title('PLS')
            plt.xlim(left=-1)
        plt.show()
    # Define PLS object with optimal number of components
    pls_opt = PLSRegression(n_components=msemin+1)
    # Fir to the entire dataset
    pls_opt.fit(X, y)
    y_c = pls_opt.predict(X)
    # Cross-validation
    y_cv = cross_val_predict(pls_opt, X, y, cv=10)
    # Calculate scores for calibration and cross-validation
    score_c = r2_score(y, y_c)
    score_cv = r2_score(y, y_cv)
    # Calculate mean squared error for calibration and cross validation
    mse_c = mean_squared_error(y, y_c)
    mse_cv = mean_squared_error(y, y_cv)
    print('R2 calib: %5.3f'  % score_c)
    print('R2 CV: %5.3f'  % score_cv)
    print('MSE calib: %5.3f' % mse_c)
    print('MSE CV: %5.3f' % mse_cv)
    # Plot regression and figures of merit
    rangey = max(y) - min(y)
    rangex = max(y_c) - min(y_c)
    # Fit a line to the CV vs response
    z = np.polyfit(y, y_c, 1)
    with plt.style.context(('ggplot')):
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.scatter(y_c, y, c='red', edgecolors='k')
        #Plot the best fit line
        ax.plot(np.polyval(z,y), y, c='blue', linewidth=1)
        #Plot the ideal 1:1 line
        ax.plot(y, y, color='green', linewidth=1)
        plt.title('$R^{2}$ (CV): '+str(score_cv))
        plt.xlabel('Predicted $^{\circ}$Milk')
        plt.ylabel('Measured $^{\circ}$Milk')
        plt.show()
    return
optimise_pls_cv(X2,y, 40, plot_components=True)


In [ ]:
# Cross validation =============================================================
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict
# Define PLS object
pls = PLSRegression(n_components=8)
# Fit
pls.fit(X, y)
# Cross-validation
y_cv = cross_val_predict(pls, X, y, cv=10)
# Calculate scores
score = r2_score(y, y_cv)
mse = mean_squared_error(y, y_cv)
print(score)
print(mse)

Classification of NIR spectra using Principal Component Analysis in Python

https://nirpyresearch.com/classification-nir-spectra-principal-component-analysis-python/

**2. Time Series Analysis**

In [ ]:
# https://finance.yahoo.com/
from pandas_datareader import data
import matplotlib.pyplot as plt
import pandas as pd


symbol = 'TSLA'   #'MSFT'  'AMZN','AAPL', 'GOOGL'
data_source='yahoo'       #'google'
start_date = '2010-01-01'
end_date = '2023-07-10'

#df = data.DataReader(symbol, data_source, start_date, end_date)
df = data.get_data_yahoo(symbol, start=start_date, end=end_date)
#df = pd.read_csv('../input/mydata/AAPL.csv')
print(df)
close = df['Close']
#print(close)
#print(close.head(10))
#print(close.describe())

# Calculate the 20 and 100 days moving averages
short_rolling = close.rolling(window=20).mean()
long_rolling = close.rolling(window=100).mean()

from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

# Plot the data
fig, ax = plt.subplots(figsize=(16,9))
ax.plot(close.index, close, label=symbol)
ax.plot(short_rolling.index,  short_rolling, label='20 days rolling')
ax.plot(long_rolling.index, long_rolling, label='100 days rolling')

ax.set_xlabel('Date')
ax.set_ylabel('Closing price ($)')
ax.legend()
plt.show()


In [ ]:
import yfinance as yf

def get_stock_price(symbol):
    stock = yf.Ticker(symbol)
    #history = stock.history(period="1d")
    history = stock.history(start="2017-01-01", end="2023-04-30")
    #return history['Close'].iloc[-1]
    return history['Close']

symbol = "AAPL"  # Replace with the stock symbol you want
#symbol = 'TSLA'   #MSFT'           #'AMZN','AAPL', 'GOOGL'
price = get_stock_price(symbol)
print(f"The current price of {symbol} is {price}")

In [ ]:
# Calculate the 20 and 100 days moving averages
short_rolling = price.rolling(window=20).mean()
long_rolling = price.rolling(window=100).mean()

from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

# Plot the data
fig, ax = plt.subplots(figsize=(16,9))
ax.plot(price.index, price, label=symbol)
ax.plot(short_rolling.index,  short_rolling, label='20 days rolling')
ax.plot(long_rolling.index, long_rolling, label='100 days rolling')

ax.set_xlabel('Date')
ax.set_ylabel('Closing price ($)')
ax.legend()
plt.show()

In [ ]:
from pandas_datareader import data

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import yfinance as yf

def get_stock_price(symbol):
    stock = yf.Ticker(symbol)
    #history = stock.history(period="1d")
    history = stock.history(start="2017-01-01", end="2023-04-30")
    #return history['Close'].iloc[-1]
    return history

symbol = "AAPL"  # Replace with the stock symbol you want
#symbol = 'TSLA'   #MSFT'           #'AMZN','AAPL', 'GOOGL'
df = get_stock_price(symbol)

close = df['Close']
#print(close)
#print(close.head(10))
#print(close.describe())

# Calculate the 20 and 100 days moving averages
short_rolling = close.rolling(window=20).mean()
long_rolling = close.rolling(window=100).mean()
#short_rolling = close.ewm(span = 20, adjust = False).mean()
#long_rolling = close.ewm(span = 100, adjust = False).mean()


from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

# Calculate the 'buy' and 'sell' signals and positions
df['Signal'] = 0.0
df['Signal'] = np.where(short_rolling > long_rolling, 1.0, 0.0)
df['Position'] = df['Signal'].diff()

# Plot the data
fig, ax = plt.subplots(figsize=(16,9))
ax.plot(close.index, close, label=symbol)
ax.plot(short_rolling.index,  short_rolling, label='20 days rolling')
ax.plot(long_rolling.index, long_rolling, label='100 days rolling')

# plot 'buy' signals
plt.plot(df[df['Position'] == 1].index,
         short_rolling[df['Position'] == 1],
         '^', markersize = 15, color = 'g', label = 'buy')

# plot 'sell' signals
plt.plot(df[df['Position'] == -1].index, 
         short_rolling[df['Position'] == -1],
         'v', markersize = 15, color = 'r', label = 'sell')

ax.set_xlabel('Date')
ax.set_ylabel('Closing price ($)')
ax.legend()
plt.show()

**Assignment:**

Modify the above code, so that it can work on your own stock data, you can either load the stock data from a website, or down it first and save it to a CSV file.

In [ ]:
#Stock Prediction with LSTM
import numpy
import matplotlib.pyplot as plt
from pandas import read_csv
import math
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# 2 34 53 3 4 5 76 34 ...
# X = 2 34 53
# Y = 3
# ...
def create_dataset(dataset, xback=1):
    dataX, dataY = [], []
    for i in range(len(dataset)-xback-1):
        a = dataset[i:(i+xback), 0]
        dataX.append(a)
        dataY.append(dataset[i + xback, 0])
    return numpy.array(dataX), numpy.array(dataY)
def getData(file, col):
    # load the dataset
    dataframe = read_csv(file,usecols=[col], engine='python')
    dataset = dataframe.values
    dataset = dataset.astype('float32')
    # normalize the dataset
    dataset = scaler.fit_transform(dataset)
    # split into train and test sets
    train_size = int(len(dataset) * ratio)
    test_size = len(dataset) - train_size
    train, test = dataset[0:train_size,:], dataset[train_size:len(dataset),:]

    trainX, trainY = create_dataset(train, xback)
    testX, testY = create_dataset(test, xback)
    # reshape input to be [samples, time steps, features]
    trainX = numpy.reshape(trainX, (trainX.shape[0], trainX.shape[1], 1))
    testX = numpy.reshape(testX, (testX.shape[0], testX.shape[1], 1))
    return trainX, trainY,testX, testY, dataset

def train(trainX, trainY,testX, testY):
    # create and fit the LSTM network
    model = Sequential()
    model.add(LSTM(4, input_shape=(xback, 1)))
    model.add(Dense(1))
    model.compile(loss='mean_squared_error', optimizer='adam')
    model.fit(trainX, trainY, epochs=10, batch_size=1, verbose=2)
    return model

def predict(model, trainX, trainY,testX, testY, dataset):
    # make predictions
    trainPredict = model.predict(trainX)
    testPredict = model.predict(testX)
    # invert predictions
    trainPredict = scaler.inverse_transform(trainPredict)
    trainY = scaler.inverse_transform([trainY])
    testPredict = scaler.inverse_transform(testPredict)
    testY = scaler.inverse_transform([testY])
    # calculate root mean squared error
    trainScore = math.sqrt(mean_squared_error(trainY[0], trainPredict[:,0]))
    print('Train Score: %.2f RMSE' % (trainScore))
    testScore = math.sqrt(mean_squared_error(testY[0], testPredict[:,0]))
    print('Test Score: %.2f RMSE' % (testScore))
    # shift train predictions for plotting
    trainPredictPlot = numpy.empty_like(dataset)
    trainPredictPlot[:, :] = numpy.nan
    trainPredictPlot[xback:len(trainPredict)+xback, :] = trainPredict
    # shift test predictions for plotting
    testPredictPlot = numpy.empty_like(dataset)
    testPredictPlot[:, :] = numpy.nan
    testPredictPlot[len(trainPredict)+(xback*2)+1:len(dataset)-1, :] = testPredict
    # plot baseline and predictions
    plt.plot(scaler.inverse_transform(dataset),'o',label='origial data')
    plt.plot(trainPredictPlot,label='predict train')
    plt.plot(testPredictPlot,label='predict test')
    plt.legend()
    plt.show()


# reshape into X=5 and Y=5+1
xback = 5
#numpy.random.seed(7)
scaler = MinMaxScaler(feature_range=(0, 1))
ratio = 0.9
file = '../input/mydata/AAPL.csv'
col = 4

trainX, trainY,testX, testY, dataset = getData(file,col)
model = train(trainX, trainY,testX, testY)
predict(model, trainX, trainY,testX, testY, dataset)


In [ ]:
# Seasonal Trend Analysis (STA)
import pandas as pd
#Global CO2 data
df = pd.read_csv('https://www.esrl.noaa.gov/gmd/webdata/ccgg/trends/co2/co2_mm_gl.csv', comment='#')
print(df.head())
dl = df['average'].values.tolist()
df = pd.Series(dl, index=pd.date_range('1-1-1980', periods=len(df), freq='M'), name = 'CO2')
print(df.head())
df.describe()


from statsmodels.tsa.seasonal import STL
stl = STL(df)
res = stl.fit()
fig = res.plot()
fig.show()

#Predition =====================================================================
from statsmodels.tsa.forecasting.stl import STLForecast
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt

data = df
stlf = STLForecast(data , ARIMA, model_kwargs={"order": (2, 1, 0)})
res = stlf.fit()

forecast = res.forecast(24)
plt.figure()
plt.plot(data)
plt.plot(forecast)
plt.show()


In [ ]:
df = pd.read_html('https://en.wikipedia.org/wiki/World_population', match='Global annual population growth')
df =df[0]
print(df)
print(len(df))
dl = df['Population'].values.tolist()
dl2=[]
for d in dl:
    dl2.append(d[0])
print(dl2)

df = pd.Series(dl2, index=pd.date_range('1-1-1951', periods=len(df), freq='m'), name = 'CO2')
print(df.head())
df.describe()

from statsmodels.tsa.seasonal import STL
stl = STL(df)
res = stl.fit()
fig = res.plot()
fig.show()

**Exercise:**

Modify the above code, so that it can work on your own time series data, you can either load the time series data from a website, or down it first and save it to a CSV file.

**3. Predictive Maintenance Analysis**

In [ ]:
# https://www.kaggle.com/c/predictive-maintenance/code
# Create a new Notebook

# Perry's Example (This version works)
# https://www.kaggle.com/code/perryxiao/predictivemaintenancetutorial

# Datasets
# https://www.kaggle.com/search?q=Predictive+Maintenance+Analysis+in%3Adatasets


**All the Lecture Notes**

**Day 1 (Chapter 1 and 2)**

https://www.kaggle.com/code/perryxiao/day-1

**Chapter 3**

https://www.kaggle.com/code/perryxiao/chapter-3

**Chapter 4**

https://www.kaggle.com/code/perryxiao/chapter-4

**Chapter 5**

https://www.kaggle.com/code/perryxiao/chapter-5

**Chapte 7**

https://www.kaggle.com/code/perryxiao/chapter-7

**Book**

Artificial Intelligence Programming with Python: From Zero to Hero

https://www.amazon.co.uk/Artificial-Intelligence-Programming-Python-Zero/dp/1119820863/ref=sr_1_1?



**Contact Information**

*Professor Perry Xiao*

*London South Bank University*

*103 Borough Road*

*London SE1 0AA*

*UK*

*Email:      perry.xiao@lsbu.ac.uk*
